[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C24_Inference_Serving_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy **把推理引擎的机制做成模拟器 + 账本**，再与朴素参考 **对拍**。

这个 notebook 做三件事：① 确认环境；② 用一个最小例子体会 **decode 为什么是带宽受限**、KV cache 为什么是显存命门；③ 立下全课的纪律——**对拍 + 不变量断言**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画吞吐/延迟曲线）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · KV cache 有多大？显存命门初体验

KV cache 的大小 ≈ `2 × 层数 × KV头数 × head_dim × 序列长 × batch × 字节/元素`（系数 2 = K 和 V）。

用一个 Llama-3-8B 量级的配置算一算：它常常比模型权重还容易成为瓶颈。

In [ ]:
def kv_cache_bytes(layers, kv_heads, head_dim, seq_len, batch, dtype_bytes=2):
    # 2 = K 和 V 两份
    return 2 * layers * kv_heads * head_dim * seq_len * batch * dtype_bytes

# Llama-3-8B 量级（GQA：32 query 头但只有 8 个 KV 头）
cfg = dict(layers=32, kv_heads=8, head_dim=128, dtype_bytes=2)  # fp16
per_token = kv_cache_bytes(seq_len=1, batch=1, **cfg)
print(f'每个 token 的 KV ≈ {per_token} 字节 = {per_token/1024:.1f} KB')

for seq, bs in [(2048, 1), (8192, 1), (8192, 32), (32768, 1)]:
    gb = kv_cache_bytes(seq_len=seq, batch=bs, **cfg) / 1e9
    print(f'  seq={seq:6d} batch={bs:3d} -> KV cache ≈ {gb:6.2f} GB')

# 关键断言：KV 随 seq 与 batch 线性增长
a = kv_cache_bytes(seq_len=1000, batch=1, **cfg)
assert kv_cache_bytes(seq_len=2000, batch=1, **cfg) == 2 * a
assert kv_cache_bytes(seq_len=1000, batch=4, **cfg) == 4 * a
print('\n✅ KV cache 随序列长、batch 线性膨胀 —— 这就是 8B 模型也能被 KV 撑爆显存的原因')

## 3 · 为什么 decode 是带宽受限？

decode 每生成 **1** 个 token，要把 **整个模型权重 + 当前全部 KV** 从显存读一遍，却只做 1 个 token 的计算。
我们算它的 **算术强度（FLOP/byte）**：远低于 GPU 的脊点 → 铁定带宽受限。

对比 prefill：一次并行处理很多 token，权重读一次服务整段，算术强度高得多。

In [ ]:
# 极简模型：参数量 P，decode 单 token 的前向 FLOPs ≈ 2P（每参数一次乘加）
# 要读的字节：权重 P*2（fp16） + KV（这里先忽略，权重已主导）
P = 8e9                       # 8B 参数
dtype_bytes = 2

def arithmetic_intensity(n_tokens, P, dtype_bytes=2):
    flops = 2 * P * n_tokens             # 处理 n 个 token 的前向
    bytes_moved = P * dtype_bytes        # 权重读一遍（batch/序列共享）
    return flops / bytes_moved

ai_decode  = arithmetic_intensity(1,   P)   # decode：一次 1 个 token
ai_prefill = arithmetic_intensity(512, P)   # prefill：一次 512 个 token
print(f'decode (1 token)   算术强度 ≈ {ai_decode:.1f} FLOP/byte')
print(f'prefill(512 token) 算术强度 ≈ {ai_prefill:.1f} FLOP/byte')
# H100 脊点 ~ 295 FLOP/byte（989 TFLOP/s ÷ 3.35 TB/s）
RIDGE = 989e12 / 3.35e12
print(f'H100 脊点         ≈ {RIDGE:.0f} FLOP/byte')
assert ai_decode < RIDGE, 'decode 应远在脊点左侧 -> 带宽受限'
assert ai_prefill > ai_decode
print('\n✅ decode 算术强度 ~1，远低于脊点 -> 带宽受限：GPU 在等数据，不在算。')
print('   推论：把多个请求 batch 在一起（权重只读一次服务整批）能极大提升 decode 吞吐 —— 这是模块 02 的动机。')

## 4 · batch 如何摊薄权重读取（continuous batching 的种子）

decode 时权重读取的代价被整个 batch **均摊**。batch 越大，分摊到每个 token 的权重字节越少 → 每 token 越快、吞吐越高。
下面这个最小账本预演模块 02 的核心收益。

In [ ]:
def bytes_per_token_decode(batch, P, dtype_bytes=2):
    '''decode 一步：权重 P*2 字节读一次，服务 batch 个 token；均摊到每 token 的权重字节。'''
    weight_bytes = P * dtype_bytes
    return weight_bytes / batch     # 均摊

P = 8e9
print(f"{'batch':>6} {'权重字节/token':>16} {'相对 batch=1':>14}")
base = bytes_per_token_decode(1, P)
for bs in [1, 2, 8, 32, 128]:
    bpt = bytes_per_token_decode(bs, P)
    print(f'{bs:>6} {bpt/1e9:>14.3f}GB {base/bpt:>13.0f}x')
assert bytes_per_token_decode(32, P) * 32 == bytes_per_token_decode(1, P)
print('\n✅ batch=32 时每 token 的权重读取降到 1/32 —— 带宽受限下，batch 几乎线性提吞吐。')
print('   但 batch 越大越吃 KV 显存（模块 01）、越抬单请求延迟（模块 02 的张力）。')

## 5 · 立纪律：对拍 + 不变量断言

本课每个「机制」都要和一个**绝对可信的朴素参考**比对，并断言其**核心不变量**。
先把这个工作流跑通：以「连续 vs 分页存 KV」为例，断言分页存回的 KV **逐位等于**连续存储（这正是模块 01 的核心不变量）。

In [ ]:
rng = np.random.default_rng(0)

# 参考：把一个序列的 KV 连续存在一个大数组里
seq_len, dim = 20, 4
kv_true = rng.standard_normal((seq_len, dim))

# 机制：分页存储——切成 block，逻辑顺序记在 block_table 里（物理可乱序）
BLOCK = 6
n_blocks = (seq_len + BLOCK - 1) // BLOCK
phys = rng.permutation(n_blocks)          # 故意把物理块打乱顺序
store = np.zeros((n_blocks, BLOCK, dim))   # 物理块池
block_table = []                           # 逻辑块 -> 物理块
for logical in range(n_blocks):
    pb = int(phys[logical])
    block_table.append(pb)
    lo = logical * BLOCK
    chunk = kv_true[lo: lo + BLOCK]
    store[pb, :len(chunk)] = chunk

# 读回：按 block_table 间接寻址，拼回逻辑顺序
def gather(block_table, store, seq_len, BLOCK):
    out = []
    for logical, pb in enumerate(block_table):
        lo = logical * BLOCK
        take = min(BLOCK, seq_len - lo)
        out.append(store[pb, :take])
    return np.concatenate(out, axis=0)

kv_read = gather(block_table, store, seq_len, BLOCK)
assert np.array_equal(kv_read, kv_true), '分页存回必须逐位等于连续存储！'
print('block_table (逻辑->物理):', block_table)
print('✅ 对拍通过：物理块乱序存放，按块表读回仍逐位等于连续 KV —— 模块 01 的核心不变量')

## 6 · 一个会贯穿全课的对拍工具

把「对拍」封装成小函数，后面每个模块都用它判定「我的机制 == 朴素参考」。它是本课所有 `assert` 背后的统一裁判。

In [ ]:
def check_allclose(name, got, ref, atol=1e-10):
    '''数值对拍：被测机制结果 vs 朴素参考。'''
    got = np.asarray(got); ref = np.asarray(ref)
    ok = np.allclose(got, ref, atol=atol)
    max_err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参考不一致！'
    return ok

def check_equal(name, got, ref):
    '''精确对拍：用于整数账本/块表等必须逐位相等的不变量。'''
    ok = np.array_equal(np.asarray(got), np.asarray(ref))
    print(f'[{name:<28}] array_equal={ok}')
    assert ok, f'{name} 与参考不一致！'
    return ok

check_allclose('gather KV vs 连续', kv_read, kv_true)
check_equal('block_table 长度', [len(block_table)], [n_blocks])
print('\n这就是全课的工作流：写机制 -> 对拍朴素参考/断言不变量 -> assert 兜底。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 里写出的每个机制（分页 KV、调度器、投机采样、radix 树、量化），都会用对拍或不变量断言来验证；机制正确则不变量成立，不变量成立则逻辑可迁移到 vLLM/SGLang/TensorRT-LLM。

**接下来五个模块**：01 PagedAttention → 02 Continuous Batching → 03 投机解码 → 04 Prefix Cache 与 PD 分离 → 05 fp8 量化。每一步都建立在前一步之上。

下一站：**模块 01 · PagedAttention 与 KV 管理**。